---
title: "Building an ML Platform"
description: "A small, low-ops ML platform built twice: first as a Docker Compose proof of concept, then ported to Azure Container Apps — same contracts, same golden path."
image: "./img/ml-platform-cover.png"
---

A step-by-step build of a **deliberately small, low-ops** MLOps platform for a
team of ML engineers who are *not* full-time platform/DevOps engineers. We use
the fewest moving parts that still deliver reproducible training, honest
evaluation, scheduled and on-demand batch/inference workflows, and observable
operations — and we add machinery only when a concrete need forces it.

The runnable source lives in **`projects/ml-platform/`**, independent of these
notebooks. Each chapter develops one slice of that project and then references
the modules and scripts it produced. The production contract the build follows
lives in **`projects/ml-platform/docs/`** (documents `00`–`08`).


## The four planes

Everything in the platform is a consequence of four planes plus a thin dashboard.

| Plane | Responsibility | Azure building block |
|---|---|---|
| **Execution** | Run every workflow as an ephemeral, image-pinned task | Azure Container Apps **Jobs** |
| **Model lifecycle** | Track experiments, register versions, store artifacts | **Self-hosted MLflow** (ACA App + Postgres + Blob) |
| **Operational state** | Record status/output/error for every run, with batch granularity | **Generic results DB** (Postgres) |
| **Serving** | Optional online HTTP inference at an exact model version | Azure Container Apps **Apps** |

There is **no bespoke control plane**: no Durable Functions, no orchestration
engine, and no application broker in the baseline. Linear multi-step workflows
are an ordinary Python script inside one Job; batch fan-out is expressed as
parent/child rows in the results DB plus a small stateless rule.


## The learning path — a phased delivery journey

The course follows the platform's **phased delivery journey** (`docs/07`). Each
phase is independently useful: the team is never blocked on machinery it does not
yet need.

| Ch | Phase | Chapter | Builds in `projects/ml-platform/` | Contract |
|----|-------|---------|-----------------------------------|----------|
| 01 | — | Overview & the golden path | — (conceptual) | `docs/00`, `docs/07` |
| 02 | 0 | Platform foundation | `infra/`, `src/mlflow_app/` | `docs/01` |
| 03 | 1 | Reproducible training & registry | `src/train_job/`, `src/ml_platform/common/` | `docs/02` |
| 04 | 2 | Results DB & batch workflows | `src/ml_platform/results/`, `src/batch_job/` | `docs/04` |
| 05 | 3 | Online serving & promotion | `src/serving_app/` | `docs/05`, `docs/06` |
| 06 | 4 | Observability & dashboard | `src/dashboard/` | `docs/06` |
| 07 | 5 | LLM release artifacts | `src/ml_platform/llm/` | `docs/03` |
| 08 | Exc. | Multi-GPU training | `src/train_aml/` | `docs/08` |
| 09 | Upg. | Broker upgrade (only if forced) | — (conditional) | `docs/04` |
| 10 | — | End-to-end integration | `deploy/` | all |

Chapters 08 and 09 are **off the critical path** — an exception track
(distributed/multi-GPU training) and a conditional upgrade (a broker), included
so the boundary is explicit but never required to ship phases 0–5.


## How the course uses the project

The **source is not authored inside the notebooks**. Each chapter:

1. States the outcome and the slice of `projects/ml-platform/docs/` it implements.
2. Builds the relevant modules/scripts under `projects/ml-platform/src/` (and
   `infra/` / `deploy/`).
3. References that source when demonstrating a stage — e.g. running an ACA Job
   from a pinned image, or loading a model version in the serving app.
4. Ends with an **Extensions** section: what the production contract asks for that
   the chapter's MVP intentionally defers, and where it is specified.

This keeps the platform a real, reviewable project you could lift out of the
course, while the notebooks stay the narrated build log.

## Prerequisites & cost

- Azure CLI with an active subscription; Python 3.11+; basic ML familiarity.
- Container builds via ACR Tasks (no local Docker required).
- Budget is small with tear-down discipline; every chapter that provisions
  resources ends by tearing them down.
